## 1) Downloads, Imports, etc.

In [ ]:
!pip install rdkit
!pip install torch
!pip install torchani
!pip install pennylane
!pip install requests aiohttp
!pip install bokeh
import pennylane as qml

# pip install rdkit-pypi torch torchani (if using ANI)
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
import numpy as np


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
  Using cached torchani-2.2.4-py3-none-any.whl (10.9 MB)
  Using cached lark_parser-0.12.0-py2.py3-none-any.whl (103 kB)

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## 2) Define a whole bunch of steps to prepare the molecules from SMILES, compute ani energies, etc. etc.


In [5]:
MILES = "O"  # Water
NAME = "water"

NCONF = 150                 # more for larger side chains
RMS_PRUNE = 0.4             # Å
KEEP_MMFF = 50              # keep this many lowest by MMFF energy
USE_ANI = True             # set True to compute ANI-2x energies here

def prepare_mol(smiles):
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)
    return m

def embed_minimize_confs(mol, nconf=NCONF):
    params = AllChem.ETKDGv3()
    params.pruneRmsThresh = -1.0   # we’ll prune ourselves
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=nconf, params=params)
    # MMFF minimize
    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94s')
    e_list = []
    for cid in conf_ids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        except Exception:
            e = 1e9
        e_list.append((cid, e))
    return e_list

def prune_by_rmsd(mol, conf_ids, rms_cut=RMS_PRUNE):
    kept = []
    for cid in conf_ids:
        keep = True
        for kc in kept:
            rms = rdMolAlign.GetBestRMS(mol, mol, prbId=cid, refId=kc)
            if rms < rms_cut:
                keep = False
                break
        if keep:
            kept.append(cid)
    return kept

def mmff_rank_and_prune(mol, conf_energy_pairs, keep=KEEP_MMFF):
    conf_energy_pairs = sorted(conf_energy_pairs, key=lambda x: x[1])
    # Take top 'keep' by energy but ensure diversity with RMSD pruning
    ranked = [cid for cid,_ in conf_energy_pairs]
    diverse = prune_by_rmsd(mol, ranked, rms_cut=RMS_PRUNE)
    # keep the best among those diverse; if too many, cap at 'keep'
    diverse_sorted = sorted(diverse, key=lambda cid: dict(conf_energy_pairs)[cid])
    return diverse_sorted[:keep]

def compute_ani_energies(mol, conf_ids, model_name='ani2x'):
    import torch, torchani
    # Load model
    model = (torchani.models.ANI2x() if model_name.lower() == 'ani2x'
             else torchani.models.ANI1ccx())
    device = torch.device('cpu')
    model = model.to(device).eval()

    # Map atomic numbers -> element symbols for TorchANI
    z2sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F', 16:'S', 17:'Cl', 35:'Br', 53:'I'}
    symbols = [z2sym[atom.GetAtomicNum()] for atom in mol.GetAtoms()]

    # TorchANI helper to build species tensor
    species = model.consts.species_to_tensor(symbols).unsqueeze(0).to(device)  # shape (1, natoms)

    energies = {}
    for cid in conf_ids:
        conf = mol.GetConformer(cid)
        coords = [[conf.GetAtomPosition(i).x,
                   conf.GetAtomPosition(i).y,
                   conf.GetAtomPosition(i).z] for i in range(mol.GetNumAtoms())]
        coordinates = torch.tensor([coords], dtype=torch.float32, device=device)  # (1, natoms, 3)
        with torch.no_grad():
            e = model((species, coordinates)).energies.item()  # Hartree
        energies[cid] = e
    return energies  # dict: confId -> Eh


def write_sdf(mol, conf_ids, fields, path):
    w = Chem.SDWriter(path)
    for cid in conf_ids:
        m = Chem.Mol(mol)
        m.SetProp("_Name", f"{NAME}_conf{cid}")
        for k,v in fields.items():
            if cid in v:
                m.SetDoubleProp(k, float(v[cid]))
        w.write(m, confId=cid)
    w.close()




### 3) Test for water

In [6]:
##########################
### Test for h2o)
##########################
mol0 = prepare_mol(MILES)
tauts = [mol0]

print(f"Found {len(tauts)} unique tautomers")
best_overall = None  # (energy_Eh, taut_idx, conf_id)

for i, taut in enumerate(tauts):
    confEs = embed_minimize_confs(taut, NCONF)
    keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)

    aniE = {}
    if USE_ANI:
        aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')

    # Report for this tautomer
    if USE_ANI and aniE:
        cid_min = min(aniE, key=lambda k: aniE[k])
        E_min = aniE[cid_min]   # Hartree
        print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
        if (best_overall is None) or (E_min < best_overall[0]):
            best_overall = (E_min, i, cid_min)
    else:
        # fall back to MMFF (NOT electronic) just so something prints
        mmffE = dict(confEs)
        cid_min = min(keep_ids, key=lambda k: mmffE[k])
        print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")

if USE_ANI and best_overall:
    E, ti, ci = best_overall
    print(f"\nGround-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")

Found 1 unique tautomers


/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/aev.py:16: UserWarning: cuaev not installed
  warnings.warn("cuaev not installed")
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/__init__.py:55: UserWarning: Dependency not satisfied, torchani.ase will not be available
  warnings.warn("Dependency not satisfied, torchani.ase will not be available")


/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -76.388251390 Eh  (conf 64)

Ground-state estimate (ANI): -76.388251390 Eh  from tautomer 0, conformer 64


### 4) estimate for molecules.

In [14]:
def estimate_gse_for_molecules(smiles_list, names, out_file="estimations_gse.txt"):
    with open(out_file, "w") as f:
        for SMILES, NAME in zip(smiles_list, names):
            print(f"\nProcessing: {NAME} ({SMILES})")
            mol0 = prepare_mol(SMILES)
            tauts = [mol0]
            best_overall = None  # (energy_Eh, taut_idx, conf_id)
            for i, taut in enumerate(tauts):
                confEs = embed_minimize_confs(taut, NCONF)
                keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)
                aniE = {}
                if USE_ANI:
                    aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')
                if USE_ANI and aniE:
                    cid_min = min(aniE, key=lambda k: aniE[k])
                    E_min = aniE[cid_min]   # Hartree
                    print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
                    if (best_overall is None) or (E_min < best_overall[0]):
                        best_overall = (E_min, i, cid_min)
                else:
                    mmffE = dict(confEs)
                    cid_min = min(keep_ids, key=lambda k: mmffE[k])
                    print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")
            if USE_ANI and best_overall:
                E, ti, ci = best_overall
                result = f"{NAME}\t{SMILES}\t{E:.9f} Eh\tTautomer {ti}\tConformer {ci}\n"
                print(f"Ground-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")
                f.write(result)
            else:
                result = f"{NAME}\t{SMILES}\tNo ANI result\n"
                f.write(result)

# Example usage:
smiles_list = [
    "O",                        # Water
    "N[C@@H](C)C(=O)O",         # Alanine (Ala)
    "N[C@@H](Cc1ccccc1)C(=O)O", # Phenylalanine (Phe)
    "N[C@@H](C)C(=O)O",         # Alanine (Ala) - duplicate, remove one
    "N[C@@H](CC(=O)O)C(=O)O",   # Aspartic acid (Asp)
    "N[C@@H](CCC(=O)O)C(=O)O",  # Glutamic acid (Glu)
    "NCC(=O)O",                 # Glycine (Gly)
    "N[C@@H](Cc1c[nH]cn1)C(=O)O", # Histidine (His)
    "CC(C)[C@H](N)C(=O)O",      # Isoleucine (Ile)
    "CC(C)C[C@H](N)C(=O)O",     # Leucine (Leu)
    "NCCCC[C@H](N)C(=O)O",      # Lysine (Lys)
    "CSCC[C@H](N)C(=O)O",       # Methionine (Met)
    "NC(CCC(N)=O)C(=O)O",       # Glutamine (Gln)
    "NC(CC(N)=O)C(=O)O",        # Asparagine (Asn)
    "N1CCC[C@H]1C(=O)O",        # Proline (Pro)
    "NC(CS)C(=O)O",             # Cysteine (Cys)
    "NC(CO)C(=O)O",             # Serine (Ser)
    "C[C@@H](O)[C@H](N)C(=O)O", # Threonine (Thr)
    "NC(Cc1c[nH]c2ccccc12)C(=O)O", # Tryptophan (Trp)
    "NC(Cc1ccc(O)cc1)C(=O)O",   # Tyrosine (Tyr)
    "CC(C)[C@H](N)C(=O)O",      # Valine (Val)
    "NC(CCCNC(N)=N)C(=O)O"      # Arginine (Arg)
]

names = [
    "water", "alanine", "phenylalanine", "aspartic_acid", "glutamic_acid", 
    "glycine", "histidine", "isoleucine", "leucine", "lysine", "methionine",
    "glutamine", "asparagine", "proline", "cysteine", "serine", "threonine", 
    "tryptophan", "tyrosine", "valine", "arginine"
]

# Corresponding amino acid codes for QMPRot lookup
amino_acid_codes = [
    None, "ala", "phe", "asp", "glu", "gly", "his", "ile", "leu", "lys", "met",
    "gln", "asn", "pro", "cys", "ser", "thr", "trp", "tyr", "val", "arg"
]

estimate_gse_for_molecules(smiles_list, names)


Processing: water (O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -76.388251391 Eh  (conf 93)
Ground-state estimate (ANI): -76.388251391 Eh  from tautomer 0, conformer 93

Processing: alanine (N[C@@H](C)C(=O)O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -323.661033051 Eh  (conf 4)
Ground-state estimate (ANI): -323.661033051 Eh  from tautomer 0, conformer 4

Processing: phenylalanine (N[C@@H](Cc1ccccc1)C(=O)O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -554.652096641 Eh  (conf 40)
Ground-state estimate (ANI): -554.652096641 Eh  from tautomer 0, conformer 40

Processing: aspartic_acid (N[C@@H](C)C(=O)O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -323

## 5) Getting 'true' values from QMPRot Database.
Theirs are calculated using the STO-3G database, so they are quite approximate...
However, this is a good way to get a starting point.

In [19]:
import pandas as pd

def get_qmprot_energies(names):
    """
    Loads energies for a list of amino acid names from the Pennylane dataset.
    Returns a pandas DataFrame with columns: name, abbreviation, energy.
    """
    records = []
    for name in names:
        try:
            data = qml.data.load("other", name=name, attributes=["abbreviation", "energy"])
            if data and len(data) > 0:
                records.append({
                    "name": name,
                    "abbreviation": getattr(data[0], "abbreviation", ""),
                    "energy": data[0].energy
                })
                print(f"{name.upper()} energy: {data[0].energy}")
            else:
                records.append({
                    "name": name,
                    "abbreviation": "",
                    "energy": None
                })
                print(f"{name.upper()}: No data found")
        except Exception as e:
            print(f"Error loading {name}: {e}")
            records.append({
                "name": name,
                "abbreviation": "",
                "energy": None
            })
    
    df = pd.DataFrame(records)
    return df

# Get reference energies for amino acids that match your SMILES list
amino_acids_subset = ["ala", "phe", "asp", "glu", "gly", "his", "ile", "leu", 
                     "lys", "met", "gln", "asn", "pro", "cys", "ser", "thr", 
                     "trp", "tyr", "val", "arg"]
print("Loading QMPRot reference energies...")

import os
import shutil

# Clear Pennylane dataset cache
def clear_qml_cache():
    """Clear corrupted QML dataset cache"""
    cache_dir = os.path.expanduser("~/.pennylane/datasets")
    if os.path.exists(cache_dir):
        print(f"Clearing cache directory: {cache_dir}")
        shutil.rmtree(cache_dir)
        print("Cache cleared. Data will be re-downloaded.")
    else:
        print("No cache directory found.")

# Clear cache and retry
clear_qml_cache()

df_energies = get_qmprot_energies(amino_acids_subset)
print("\nQMPRot Energy DataFrame:")
print(df_energies)

# Save to file for later comparison
df_energies.to_csv("qmprot_reference_energies.csv", index=False)
print("\nReference energies saved to 'qmprot_reference_energies.csv'")

Loading QMPRot reference energies...
No cache directory found.
ALA energy: -317.69135
PHE energy: -544.43743
ASP energy: -502.76713
GLU energy: -541.3498
GLY energy: -279.11151
HIS energy: -538.52442
ILE energy: -432.82708
LEU energy: -433.42225
LYS energy: -487.7406
MET energy: -788.02138
GLN energy: -521.82178
ASN energy: -483.2392307
PRO energy: -393.7002
CYS energy: -710.8573
Error loading ser: Unable to synchronously open file (truncated file: eof = 96, sblock->base_addr = 0, stored_eof = 2048)
THR energy: -430.09637
Error loading trp: "Unable to synchronously open object (object 'abbreviation' doesn't exist)"
Error loading tyr: "Unable to synchronously open object (object 'abbreviation' doesn't exist)"
VAL energy: -394.84749
ARG energy: -595.17254

QMPRot Energy DataFrame:
   name abbreviation      energy
0   ala          ala -317.691350
1   phe          phe -544.437430
2   asp          asp -502.767130
3   glu          glu -541.349800
4   gly          gly -279.111510
5   his     

In [16]:
# ### sample code to just get one
# # from qmprot
# data_cys = qml.data.load("other", name="cys", attributes=["abbreviation", "energy"])
# print("Cysteine energy:", data_cys[0].energy)
# print(dir(data_cys[0]))

## 6) Comparison code

In [21]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource
import numpy as np

def compare_classical_vs_qmprot():
    """
    Creates a Bokeh plot comparing classical ANI estimates vs QMPRot reference values
    """
    # Load the classical estimates from file
    classical_data = {}
    try:
        with open("estimations_gse.txt", "r") as f:
            for line in f:
                if line.strip() and not line.startswith("Name"):
                    parts = line.strip().split("\t")
                    if len(parts) >= 3 and "No ANI result" not in parts[2]:
                        name = parts[0]
                        energy = float(parts[2].replace(" Eh", ""))
                        classical_data[name] = energy
    except FileNotFoundError:
        print("Run estimate_gse_for_molecules() first to generate estimations_gse.txt")
        return
    
    print(f"Found {len(classical_data)} classical estimates")
    
    # Get QMPRot reference data with error handling
    amino_acid_names = ["ala", "phe", "asp", "glu", "gly", "his", "ile", "leu", "lys", "met",
                       "gln", "asn", "pro", "cys", "ser", "thr", "trp", "tyr", "val", "arg"]
    
    # Load QMPRot data with individual error handling
    qmprot_data = {}
    for aa in amino_acid_names:
        try:
            data = qml.data.load("other", name=aa, attributes=["abbreviation", "energy"])
            if data and len(data) > 0 and data[0].energy is not None:
                qmprot_data[aa] = float(data[0].energy)
                print(f"✓ Loaded {aa}: {data[0].energy}")
            else:
                print(f"✗ No data for {aa}")
        except Exception as e:
            print(f"✗ Error loading {aa}: {e}")
    
    print(f"Found {len(qmprot_data)} QMPRot reference values")
    
    # Create mapping from full names to abbreviations for matching
    name_mapping = {
        "alanine": "ala", "phenylalanine": "phe", "aspartic_acid": "asp", 
        "glutamic_acid": "glu", "glycine": "gly", "histidine": "his", 
        "isoleucine": "ile", "leucine": "leu", "lysine": "lys", "methionine": "met",
        "glutamine": "gln", "asparagine": "asn", "proline": "pro", 
        "cysteine": "cys", "serine": "ser", "threonine": "thr", 
        "tryptophan": "trp", "tyrosine": "tyr", "valine": "val", "arginine": "arg"
    }
    
    # Match up the data - only include pairs where both values exist
    classical_vals = []
    qmprot_vals = []
    labels = []
    
    for full_name, abbrev in name_mapping.items():
        if full_name in classical_data and abbrev in qmprot_data:
            classical_vals.append(classical_data[full_name])
            qmprot_vals.append(qmprot_data[abbrev])
            labels.append(full_name.capitalize())
            print(f"Matched {full_name}: ANI={classical_data[full_name]:.6f}, QMPRot={qmprot_data[abbrev]:.6f}")
    
    if len(classical_vals) == 0:
        print("No matching data found between classical estimates and QMPRot values")
        return
    
    print(f"\nUsing {len(classical_vals)} matched pairs for comparison")
    
    # Create Bokeh plot
    output_notebook()
    
    p = figure(
        width=600, height=600,
        title="Classical ANI vs QMPRot Ground State Energies",
        x_axis_label="ANI-2x Energy (Hartree)",
        y_axis_label="QMPRot Energy (Hartree)"
    )
    
    # Create data source for hover (using updated Bokeh syntax)
    differences = [c - q for c, q in zip(classical_vals, qmprot_vals)]
    source = ColumnDataSource(data=dict(
        x=classical_vals,
        y=qmprot_vals,
        labels=labels,
        diff=differences
    ))
    
    # Add scatter points (fixed deprecated syntax)
    scatter = p.scatter('x', 'y', size=10, color='blue', alpha=0.7, 
                       source=source, legend_label="Amino Acids")
    
    # Add hover tool
    hover = HoverTool(
        tooltips=[
            ("Molecule", "@labels"),
            ("ANI-2x", "@x{0.000000}"),
            ("QMPRot", "@y{0.000000}"),
            ("Difference", "@diff{0.000000}")
        ],
        renderers=[scatter]
    )
    p.add_tools(hover)
    
    # Add perfect correlation line
    if len(classical_vals) > 0:
        min_val = min(min(classical_vals), min(qmprot_vals))
        max_val = max(max(classical_vals), max(qmprot_vals))
        p.line([min_val, max_val], [min_val, max_val], 
               line_color='red', line_dash='dashed', 
               legend_label="Perfect Correlation")
    
    p.legend.location = "top_left"
    
    # Print correlation statistics (with safety checks)
    if len(classical_vals) > 1:
        correlation = np.corrcoef(classical_vals, qmprot_vals)[0, 1]
        mae = np.mean(np.abs(differences))
        rmse = np.sqrt(np.mean(np.array(differences)**2))
        
        print(f"\nCorrelation Statistics:")
        print(f"Correlation coefficient: {correlation:.4f}")
        print(f"Mean Absolute Error: {mae:.6f} Hartree")
        print(f"Root Mean Square Error: {rmse:.6f} Hartree")
        print(f"Number of molecules: {len(classical_vals)}")
    else:
        print(f"\nInsufficient data for correlation analysis (only {len(classical_vals)} points)")
    
    show(p)
    return p

# Run the comparison
compare_classical_vs_qmprot()

Found 21 classical estimates
✓ Loaded ala: -317.69135
✓ Loaded phe: -544.43743
✓ Loaded asp: -502.76713
✓ Loaded glu: -541.3498
✓ Loaded gly: -279.11151
✓ Loaded his: -538.52442
✓ Loaded ile: -432.82708
✓ Loaded leu: -433.42225
✓ Loaded lys: -487.7406
✓ Loaded met: -788.02138
✓ Loaded gln: -521.82178
✓ Loaded asn: -483.2392307
✓ Loaded pro: -393.7002
✓ Loaded cys: -710.8573
✗ Error loading ser: Unable to synchronously open file (truncated file: eof = 96, sblock->base_addr = 0, stored_eof = 2048)
✓ Loaded thr: -430.09637
✗ Error loading trp: "Unable to synchronously open object (object 'abbreviation' doesn't exist)"
✗ Error loading tyr: "Unable to synchronously open object (object 'abbreviation' doesn't exist)"
✓ Loaded val: -394.84749
✓ Loaded arg: -595.17254
Found 17 QMPRot reference values
Matched alanine: ANI=-323.661033, QMPRot=-317.691350
Matched phenylalanine: ANI=-554.652097, QMPRot=-544.437430
Matched aspartic_acid: ANI=-323.661033, QMPRot=-502.767130
Matched glutamic_acid: ANI

Loading BokehJS ...


Correlation Statistics:
Correlation coefficient: -0.0556
Mean Absolute Error: 142.930742 Hartree
Root Mean Square Error: 180.230502 Hartree
Number of molecules: 17


figure(id='p1300', ...)